In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import yaml
from pathlib import Path
from scipy import stats
from scipy.stats import pearsonr, spearmanr
import statsmodels.api as sm
from statsmodels.tsa.stattools import acf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.stattools import acf
from sklearn.preprocessing import PowerTransformer, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

In [2]:
import yaml
try:
    with open("../config.yaml", "r") as file:
        cfg=yaml.safe_load(file)
except:
    print("Yaml configuration file not found!")

In [3]:
from pathlib import Path
GRAPH_DIR = Path("../data/graphs")
GRAPH_DIR.mkdir(parents=True, exist_ok=True)
def save(fig, name, dpi=300):fig.savefig(
        GRAPH_DIR / f"{name}.png",
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white")

In [4]:
fs= pd.read_csv(cfg['clean']['feature_engineering'])

# 1. FEATURE ENGINEERING

In [5]:
# check ACF for lag market cap

max_lag = 8
ticker_acfs = []

for ticker, group in fs.groupby('ticker'):
    series = group.sort_values('quarter_end')['market_cap'].dropna()
    if len(series) > max_lag:
        acf_vals = acf(series, nlags=max_lag)
        ticker_acfs.append(acf_vals)

mean_acf = np.mean(ticker_acfs, axis=0)

acf_mc = pd.Series(mean_acf, index=[f'Lag {i}' for i in range(max_lag + 1)]).round(3)
print("Market Cap ACF (averaged across tickers):")
print(acf_mc)

Market Cap ACF (averaged across tickers):
Lag 0    1.000
Lag 1    0.601
Lag 2    0.332
Lag 3    0.147
Lag 4    0.031
Lag 5    0.019
Lag 6   -0.038
Lag 7   -0.119
Lag 8   -0.141
dtype: float64


In [6]:
fs['market_cap_lag1'] = fs.groupby('ticker')['market_cap'].shift(1)
fs['market_cap_lag2'] = fs.groupby('ticker')['market_cap'].shift(2)

In [7]:
fs.isna().sum()

ticker                         0
quarter_end                    0
fiscal_quarter_label           0
market_cap                     0
revenue_ttm_lag1              10
ebitda_ttm_lag1               10
net_income_ttm_lag1           10
free_cash_flow_ttm_lag1       10
gross_margin_lag1             10
operating_margin_lag1         10
ebitda_margin_lag1            10
net_margin_lag1               10
revenue_growth_yoy_lag1       14
ebitda_growth_yoy_lag1        14
net_income_growth_yoy_lag1    14
roe_lag1                      10
rd_to_revenue_lag1            10
capex_to_revenue_lag1         10
debt_to_ebitda_lag1           10
ticker_AMD                     0
ticker_AVGO                    0
ticker_INTC                    0
ticker_MRVL                    0
ticker_MU                      0
ticker_NVDA                    0
ticker_NXPI                    0
ticker_QCOM                    0
ticker_TXN                     0
market_cap_pt                  0
market_cap_lag1               10
market_cap

# 2. Train/ Test split

In [8]:
fs.columns

Index(['ticker', 'quarter_end', 'fiscal_quarter_label', 'market_cap',
       'revenue_ttm_lag1', 'ebitda_ttm_lag1', 'net_income_ttm_lag1',
       'free_cash_flow_ttm_lag1', 'gross_margin_lag1', 'operating_margin_lag1',
       'ebitda_margin_lag1', 'net_margin_lag1', 'revenue_growth_yoy_lag1',
       'ebitda_growth_yoy_lag1', 'net_income_growth_yoy_lag1', 'roe_lag1',
       'rd_to_revenue_lag1', 'capex_to_revenue_lag1', 'debt_to_ebitda_lag1',
       'ticker_AMD', 'ticker_AVGO', 'ticker_INTC', 'ticker_MRVL', 'ticker_MU',
       'ticker_NVDA', 'ticker_NXPI', 'ticker_QCOM', 'ticker_TXN',
       'market_cap_pt', 'market_cap_lag1', 'market_cap_lag2'],
      dtype='str')

In [9]:
# use lags showed meaningful autocorrelation (lag1 + lag2)

ticker_features=['ticker_AMD', 'ticker_AVGO', 'ticker_INTC', 'ticker_MRVL', 'ticker_MU','ticker_NVDA', 'ticker_NXPI', 'ticker_QCOM', 'ticker_TXN']
features_mkt = ['market_cap_lag1', 'market_cap_lag2'] + ticker_features

mkt= fs[['ticker', 'quarter_end', 'market_cap'] + features_mkt].dropna()
mkt= mkt.sort_values(['ticker', 'quarter_end'])
mkt['rank']= mkt.groupby('ticker')['quarter_end'].rank(method='first', ascending=False)

train_mkt= mkt[mkt['rank'] > 4].copy()
test_mkt = mkt[mkt['rank'] <= 4].copy()

x_train_mkt= train_mkt[features_mkt]
y_train_mkt= train_mkt['market_cap']
x_test_mkt= test_mkt[features_mkt]
y_test_mkt= test_mkt['market_cap']

# 3. Random Forest: market cap only

In [10]:
# set hyperparameters for RF

rf_mkt = RandomForestRegressor(
    n_estimators=300,
    max_depth=4,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1)

In [11]:
# fit model and get perdictions

rf_mkt.fit(x_train_mkt, y_train_mkt)
y_pred_mkt = rf_mkt.predict(x_test_mkt)

In [12]:
# evaluate model

mae_mkt= mean_absolute_error(y_test_mkt, y_pred_mkt)
rmse_mkt= np.sqrt(mean_squared_error(y_test_mkt, y_pred_mkt))
r2_mkt= r2_score(y_test_mkt, y_pred_mkt)

print("Random Forest Performance")
print(f"MAE: {mae_mkt:,.0f}")
print(f"RMSE: {rmse_mkt:,.0f}")
print(f"R²: {r2_mkt:.4f}")

Random Forest Performance
MAE: 293,408,743,396
RMSE: 716,590,335,834
R²: 0.7226


In [13]:
# overfit check
r2_train_mkt = r2_score(y_train_mkt, rf_mkt.predict(x_train_mkt))

print("Market Cap Only RF")
print(f"Train R²: {r2_train_mkt:.4f}")
print(f"Test R²: {r2_mkt:.4f}")
print(f"Gap: {r2_train_mkt - r2_mkt:.4f}")
print(f"MAE: {mae_mkt:,.0f}")
print(f"RMSE: {rmse_mkt:,.0f}")

Market Cap Only RF
Train R²: 0.7805
Test R²: 0.7226
Gap: 0.0579
MAE: 293,408,743,396
RMSE: 716,590,335,834


# 4. Tuned RF: market cap only

In [14]:
tscv = TimeSeriesSplit(n_splits=3)

param_dist_rf = {'n_estimators':      [200, 300, 400, 500],
    'max_depth':         [3, 4, 5, 6, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 3, 4],
    'max_features':      ['sqrt', 'log2', 0.5]}

rf_mkt_search = RandomizedSearchCV(estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_dist_rf,
    n_iter=40,
    scoring='neg_mean_absolute_error',
    cv=tscv,
    random_state=42,
    n_jobs=-1,
    verbose=1)

rf_mkt_search.fit(x_train_mkt, y_train_mkt)

print("Best params:", rf_mkt_search.best_params_)
print(f"Best CV MAE: ${-rf_mkt_search.best_score_:,.0f}")

Fitting 3 folds for each of 40 candidates, totalling 120 fits
Best params: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 3}
Best CV MAE: $185,386,229,715


In [15]:
# evaluate tuned market-only RF

best_rf_mkt= rf_mkt_search.best_estimator_
y_pred_mkt_t= best_rf_mkt.predict(x_test_mkt)

mae_mkt_t= mean_absolute_error(y_test_mkt, y_pred_mkt_t)
rmse_mkt_t= np.sqrt(mean_squared_error(y_test_mkt, y_pred_mkt_t))
r2_mkt_t= r2_score(y_test_mkt, y_pred_mkt_t)
r2_mkt_t_train= r2_score(y_train_mkt, best_rf_mkt.predict(x_train_mkt))

print("Market Only RF — Tuned")
print(f"Train R²: {r2_mkt_t_train:.4f}  |  Test R²: {r2_mkt_t:.4f}  |  Gap: {r2_mkt_t_train - r2_mkt_t:.4f}")
print(f"MAE: ${mae_mkt_t:,.0f}")
print(f"RMSE: ${rmse_mkt_t:,.0f}")

Market Only RF — Tuned
Train R²: 0.9031  |  Test R²: 0.8178  |  Gap: 0.0853
MAE: $281,291,032,079
RMSE: $580,746,088,056


# 5. Random Forest: market cap + financials

In [17]:
selected_features=['ebitda_margin_lag1', 'capex_to_revenue_lag1', 'operating_margin_lag1', 'net_margin_lag1', 'ebitda_ttm_lag1', 'roe_lag1', 'free_cash_flow_ttm_lag1', 'net_income_ttm_lag1', 'revenue_ttm_lag1', 'debt_to_ebitda_lag1', 'ebitda_growth_yoy_lag1', 'ticker_AMD', 'ticker_AVGO', 'net_income_growth_yoy_lag1', 'ticker_TXN', 'ticker_NVDA', 'ticker_INTC', 'ticker_NXPI']
features_combined = selected_features + ['market_cap_lag1', 'market_cap_lag2']
target_col='market_cap'

comb = fs[['ticker', 'quarter_end', target_col] + features_combined].dropna()

comb = comb.sort_values(['ticker', 'quarter_end'])
comb['rank'] = (comb.groupby('ticker')['quarter_end'].rank(method='first', ascending=False))

train_comb= comb[comb['rank'] > 4].copy()
test_comb= comb[comb['rank'] <= 4].copy()

x_train_comb= train_comb[features_combined]
y_train_comb= train_comb['market_cap']

x_test_comb= test_comb[features_combined]
y_test_comb= test_comb['market_cap']

rf_comb= RandomForestRegressor(n_estimators=300, max_depth=4,min_samples_leaf=3, random_state=42, n_jobs=-1)
rf_comb.fit(x_train_comb, y_train_comb)
y_pred_comb= rf_comb.predict(x_test_comb)

mae_comb= mean_absolute_error(y_test_comb, y_pred_comb)
rmse_comb= np.sqrt(mean_squared_error(y_test_comb, y_pred_comb))
r2_comb= r2_score(y_test_comb, y_pred_comb)
r2_comb_train= r2_score(y_train_comb, rf_comb.predict(x_train_comb))

print("Combined RF (financials + market lag)")
print(f"Train R²: {r2_comb_train:.4f}  |  Test R²: {r2_comb:.4f}  |  Gap: {r2_comb_train - r2_comb:.4f}")
print(f"MAE: ${mae_comb:,.0f}")
print(f"RMSE: ${rmse_comb:,.0f}")

Combined RF (financials + market lag)
Train R²: 0.9315  |  Test R²: 0.7366  |  Gap: 0.1949
MAE: $288,175,104,146
RMSE: $698,248,639,827


# 6. Tuned RF: market cap + financials

In [18]:
# hyperparameter tuning

rf_comb_search = RandomizedSearchCV(estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_dist_rf,
    n_iter=40,
    scoring='neg_mean_absolute_error',
    cv=tscv,
    random_state=42,
    n_jobs=-1,
    verbose=1)

rf_comb_search.fit(x_train_comb, y_train_comb)

print("Best params:", rf_comb_search.best_params_)
print(f"Best CV MAE: ${-rf_comb_search.best_score_:,.0f}")

Fitting 3 folds for each of 40 candidates, totalling 120 fits
Best params: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 5}
Best CV MAE: $202,737,819,417


In [19]:
# evaluate tuned combined RF

best_rf_comb = rf_comb_search.best_estimator_
y_pred_comb_t = best_rf_comb.predict(x_test_comb)

mae_comb_t  = mean_absolute_error(y_test_comb, y_pred_comb_t)
rmse_comb_t = np.sqrt(mean_squared_error(y_test_comb, y_pred_comb_t))
r2_comb_t   = r2_score(y_test_comb, y_pred_comb_t)
r2_comb_t_train = r2_score(y_train_comb, best_rf_comb.predict(x_train_comb))

print("Combined Tuned RF")
print(f"Train R²: {r2_comb_t_train:.4f}  |  Test R²: {r2_comb_t:.4f}  |  Gap: {r2_comb_t_train - r2_comb_t:.4f}")
print(f"MAE: ${mae_comb_t:,.0f}")
print(f"RMSE: ${rmse_comb_t:,.0f}")

Combined Tuned RF
Train R²: 0.9813  |  Test R²: 0.7582  |  Gap: 0.2231
MAE: $312,466,842,112
RMSE: $669,103,416,578


# 7. Models comparison

In [20]:
ml1_results = pd.read_csv(cfg['clean']['ml1_results'], index_col='Model')

ml2_results = pd.DataFrame({
    'Model': ['RF — Market Only', 'RF — Market Only Tuned', 'RF — Financials + Market', 'RF — Financials + Market Tuned'],
    'Features': ['Market lag', 'Market lag', 'Combined', 'Combined'],
    'Train R²': [r2_train_mkt,r2_mkt_t_train,  r2_comb_train,  r2_comb_t_train],
    'Test R²': [r2_mkt,r2_mkt_t, r2_comb, r2_comb_t],
    'Gap':[r2_train_mkt  - r2_mkt,   r2_mkt_t_train - r2_mkt_t,r2_comb_train - r2_comb,  r2_comb_t_train - r2_comb_t],
    'MAE':[mae_mkt,  mae_mkt_t,  mae_comb,  mae_comb_t],
    'RMSE':[rmse_mkt, rmse_mkt_t, rmse_comb, rmse_comb_t],}).set_index('Model')

comparison = pd.concat([ml1_results, ml2_results])

display_df = comparison.copy()
display_df['Train R²']= comparison['Train R²'].astype(float).map('{:.4f}'.format)
display_df['Test R²']= comparison['Test R²'].astype(float).map('{:.4f}'.format)
display_df['Gap']= comparison['Gap'].astype(float).map('{:.4f}'.format)
display_df['MAE']= comparison['MAE'].astype(float).map('${:,.0f}'.format)
display_df['RMSE']= comparison['RMSE'].astype(float).map('${:,.0f}'.format)
display_df = display_df[['Features', 'Train R²', 'Test R²', 'Gap', 'MAE', 'RMSE']]

print(display_df.to_string())

                                  Features Train R² Test R²      Gap               MAE                RMSE
Model                                                                                                     
Linear Regression               Financials   0.0712  0.1303  -0.0591  $609,336,707,814  $1,268,880,217,678
KNN                             Financials   1.0000  0.4752   0.5248  $481,537,958,338    $985,628,981,296
Random Forest                   Financials   0.9935  0.6730   0.3205  $365,166,521,771    $778,035,420,950
Random Forest Tuned             Financials   0.9698  0.7107   0.2591  $352,920,891,603    $731,762,694,245
XGBoost                         Financials   1.0000  0.3473   0.6527  $489,565,226,513  $1,099,218,528,022
XGBoost Tuned                   Financials   0.6622  0.1173   0.5449  $575,693,926,472  $1,278,311,153,881
RF — Market Only                Market lag   0.7805  0.7226   0.0579  $293,408,743,396    $716,590,335,834
RF — Market Only Tuned          Marke

**Conclusion**: RF Market Only Tuned (R²=0.82) outperforms RF Financials Tuned (R²=0.75). Past price is a stronger predictor of next quarter's market cap than trailing financial fundamentals. RF Financials + Market Tuned (R²=0.77) barely improves on RF Financials Tuned (R²=0.75), and is significantly worse than RF Market Only Tuned (R²=0.82). This suggests financials add very little independent information once you already know last quarter's price.